# UJIAN TENGAH SEMESTER (UTS) GASAL 2026/2027
## PROGRAM STUDI S1 SAINS DATA - FMIPA UNIVERSITAS NEGERI SURABAYA

* **Mata Kuliah**: Graf Sains Data (3 SKS)
* **Materi**: Python Driver untuk Neo4J, Graph Data Science, Vector Space Modelling (VSM), dan Sistem Rekomendasi (KNN & Analytics)
* **Nama Mahasiswa**: Muhammad Naza Firmansyah
* **NIM**: 24031554164
* **Sifat Ujian**: Individu
* **Dosen Pengampu**: TIM Dosen Graf Sains Data UNESA

---
### Ringkasan Pengerjaan:
Notebook ini memuat dokumentasi lengkap pengerjaan UTS Graf Sains Data:
1. **Tugas Pendahuluan**: Persiapan lingkungan (`uv`/`venv`), eksplorasi driver Neo4J Python API, jawaban *checkpoint*, dan skrip impor RetailRocket (`events.csv`).
2. **Soal 2.1 Wrangling dan EDA (45 Poin)**: Pemilihan 2 periode waktu unik berbasis NIM (16-17 Juli 2015 untuk Fold Training P1, 18-19 Juli 2015 untuk Fold Testing P2), filter timestamp & ekspor file `.csv`, query komposisi subgraph di Neo4j via Python API, pemodelan VSM simpul User & Item, analisis komparasi proyeksi, perhitungan similarity Cosine pada Python, penulisan kembali relasi `SIMILAR` ke Neo4j via Cypher SQL, dan visualisasi kondisi *sebelum vs sesudah*.
3. **Soal 2.2 Analytics (55 Poin)**: Pemilihan user secara acak dan *reproducible* (`seed` berbasis NIM), algoritma rekomendasi Item-based KNN ($k=5$), pemilihan 3 item yang diiklankan, serta evaluasi metrik `Hit@5` dan `NDCG@5` secara analitis dan matematis terhadap *ground truth* data uji P2.

---
# 1. Tugas Pendahuluan

## 1.1 Persiapan dan Perkenalan Lingkungan
Pengembangan dilakukan dalam lingkungan virtual environment terisolasi (`myvenv`) menggunakan Python 3.10 dan manajer paket modern `uv` untuk memastikan isolasi dependensi paket dari *local environment*.

Dependensi yang diinstal meliputi:
* `neo4j>=5.0` (kompatibel dengan driver Neo4j 6.4 API: `execute_query` dan `RoutingControl`)
* `pandas` & `numpy` (data wrangling & manipulasi matriks)
* `scipy` & `scikit-learn` (sparse matrix, VSM, dan perhitungan cosine similarity)
* `networkx` & `matplotlib` / `seaborn` (pemodelan graf, topologi, dan visualisasi)
* `tqdm` (progress bar untuk batch import)

In [1]:
# Verifikasi instalasi paket dan versi library
import os
import sys
import neo4j
import pandas as pd
import numpy as np
import scipy
import sklearn
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Python Version: {sys.version}")
print(f"Neo4j Driver Version: {neo4j.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"NetworkX Version: {nx.__version__}")
print(f"Scikit-Learn Version: {sklearn.__version__}")

Python Version: 3.10.19 (main, Dec  9 2025, 18:58:23) [MSC v.1944 64 bit (AMD64)]
Neo4j Driver Version: 6.4.1
Pandas Version: 2.3.3
NumPy Version: 2.2.6
NetworkX Version: 3.4.2
Scikit-Learn Version: 1.7.2


## 1.2 Mempelajari Penggunaan Driver Neo4J
Berikut adalah implementasi tutorial driver Neo4J untuk menginisialisasi string koneksi, pembuatan objek node beserta relasinya, dan pembacaan data kembali.

In [2]:
from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://localhost:7687"
AUTH = ("neo4j", "password")

def add_friend(driver, name, friend_name):
    """Membuat simpul Person dan relasi KNOWS"""
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_="neo4j",
    )

def print_friends(driver, name):
    """Membaca dan menampilkan teman dari seorang Person"""
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_="neo4j", routing_=RoutingControl.READ,
    )
    print(f"Teman dari {name}:")
    for record in records:
        print(f" - {record['friend.name']}")

# Jalankan pengujian
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")

Teman dari Arthur:
 - Guinevere
 - Lancelot
 - Merlin


### Pertanyaan Pendahuluan (Checkpoint)

#### 1. Apa nama database yang digunakan pada tutorial ini?
> **Jawaban:**  
> Nama database yang digunakan adalah **`neo4j`**. Hal ini terlihat secara eksplisit pada argumen parameter `database_="neo4j"` di pemanggilan fungsi `driver.execute_query(...)`. Pada instalasi Neo4j, `neo4j` merupakan nama database graf *default* tempat data disimpan dan dikueri.

#### 2. Sebutkan akses lain (seperti sintaks DDL/DML CYPHER) yang dapat digunakan mengakses database lain dalam 1 instance!
> **Jawaban:**  
> Pada instance Neo4j multi-database, akses ke database lain dapat dilakukan melalui beberapa mekanisme:
> 1. **Klausul Cypher `USE <database_name>`**: Dalam sesi kueri multi-database, klausul `USE` menentukan konteks database target, contohnya:
>    ```cypher
>    USE system
>    SHOW DATABASES;
>    ```
>    atau `USE myotherdb MATCH (n) RETURN count(n);`.
> 2. **Parameter Sesi pada Python Driver**: Mengatur parameter `database` saat membuka session:
>    ```python
>    session = driver.session(database="system")
>    # atau
>    session = driver.session(database="database_lain")
>    ```
> 3. **Perintah DDL Administratif Cypher (dijalankan di database `system`)**:
>    * `SHOW DATABASES;` : Menampilkan seluruh database dalam instance.
>    * `CREATE DATABASE nama_db;` : Membuat database baru.
>    * `STOP DATABASE nama_db;` & `START DATABASE nama_db;` : Mengatur status aktif database.
>    * `DROP DATABASE nama_db;` : Menghapus database.

#### 3. Apa jenis graf pertemanan yang telah dibuat di sini?
> **Jawaban:**  
> Graf yang dibuat adalah **Homogeneous Directed Graph** (Graf Berarah Homogen).  
> * **Homogen**: Semua simpul memiliki label yang seragam (`:Person`) dan semua relasi bertipe sama (`[:KNOWS]`).  
> * **Berarah (*Directed*)**: Hubungan bersifat satu arah dari simpul sumber ke target `(a)-[:KNOWS]->(friend)`.  
> * **Topologi**: Membentuk topologi **Ego-Network (Star Graph)** dengan simpul `Arthur` sebagai simpul pusat (*hub* / ego) dengan *in-degree* = 0 dan *out-degree* = 3 menuju `Guinevere`, `Lancelot`, dan `Merlin`.

In [3]:
# Pembersihan konten database tutorial (CYPHER Total Reset)
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.execute_query("MATCH (n) DETACH DELETE n;", database_="neo4j")
    print("Database tutorial berhasil di-reset bersih (0 nodes).")

Database tutorial berhasil di-reset bersih (0 nodes).


## 1.3 Melakukan Proses Impor Data dari RetailRocket ke Database Neo4J
Dataset yang digunakan berasal dari *Roman Zykov (Retailrocket e-commerce dataset)* yang berisi interaksi pengguna di platform belanja daring.

### Rincian Fitur Dataset:
* `timestamp`: Waktu interaksi dalam format Unix Epoch (milidetik).
* `visitorid`: ID unik pengguna / pengunjung situs (*User*).
* `event`: Jenis interaksi pengguna, terdiri dari tiga nilai:
  1. `view` (pengguna melihat halaman produk)
  2. `addtocart` (pengguna memasukkan produk ke keranjang belanja)
  3. `transaction` (pengguna melakukan pembelian/transaksi produk)
* `itemid`: ID unik produk (*Item*).
* `transactionid`: ID transaksi pembelian, hanya terisi jika `event == 'transaction'`, bernilai `NaN` jika lainnya.

Berikut adalah fungsi impor batch yang mengimplementasikan `UNWIND`, `MERGE`, dan `CALL` subquery sesuai spesifikasi soal.

In [4]:
def create_constraints(session):
    """Membuat constraint UNIQUE pada Visitor(visitorId) dan Item(itemId) agar MERGE berjalan cepat"""
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE")
    session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (i:Item) REQUIRE i.itemId IS UNIQUE")
    print("Unique constraints berhasil dipastikan.")

def import_batch(tx, batch):
    """Cypher query batch import sesuai template panduan soal"""
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})
    WITH v, i, row
    CALL {
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)
        UNION ALL
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)
        UNION ALL
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)

---
# 2. Soal UTS

## 2.1 Wrangling dan EDA (Maks. skor: 45)

### Soal 2.1.1: Penentuan 2 Periode Waktu sebagai Filter Data Interaksi
Ketentuan: *Kombinasi periode yg dipilih per individu tidak boleh sama dalam 1 kelas.*

**Identitas Mahasiswa:**
* Nama: **Muhammad Naza Firmansyah**
* NIM: **24031554164** (3 digit terakhir: `164`)

**Penentuan Periode Waktu Sesuai Konvensi Kelas:**
Mengikuti metode holdout time-series split standar di kelas yang membagi keseluruhan rentang dataset RetailRocket (3 Mei 2015 hingga 18 September 2015), titik potong (*cut-off date*) ditentukan secara unik berdasarkan digit belakang NIM `164` (tanggal **16 Juli 2015**):
* **Fold Training (P1)**: **3 Mei 2015 s.d. 16 Juli 2015**
  * UNIX Timestamp Mulai: `1430622004384` (3 Mei 2015, 03:00:04 UTC)
  * UNIX Timestamp Selesai: `1437091183647` (16 Juli 2015, 23:59:43 UTC)
* **Fold Testing (P2)**: **17 Juli 2015 s.d. 18 September 2015**
  * UNIX Timestamp Mulai: `1437091199857` (17 Juli 2015, 00:00:00 UTC)
  * UNIX Timestamp Selesai: `1442545187788` (18 September 2015, 02:59:47 UTC)

**Keunggulan dan Rasionalisasi Pemilihan Periode Ini:**
1. **Unik dan Personal**: Titik potong tanggal 16 Juli diambil dari NIM `164`. Posisi tanggal 16 Juli ini sangat ideal dan unik di antara rekan sekelas (misal: Umi di 14 Juli dan Nadine di 21 Juli).
2. **Karakteristik Data Lengkap**: Membagi seluruh interaksi historis platform e-commerce (total >2,75 juta baris interaksi) menjadi masa pelatihan historis dan masa pengujian masa depan.
3. **Penyusunan Subgraph Representatif (Soal 2.1.3)**: Sesuai instruksi soal No. 3 (*"Lakukan query untuk mendapatkan komposisi dari **subgraph yang dibuat**..."*), dari partisi waktu P1 dan P2 yang masif ini, diekstraksi *subgraph interaksi aktif* di sekitar titik potong 16 Juli (16-17 Juli untuk pelatihan P1 dan 18-19 Juli untuk pengujian P2) dan disimpan ke `events_filtered.csv`. Hal ini menjaga efisiensi komputasi VSM, mencegah Out-of-Memory (OOM) pada RAM laptop saat perkalian matriks graf, serta menghasilkan visualisasi NetworkX yang rapi dan informatif.